# SpikedLM — longer Shakespeare run

Train the JAX **attention + Spiking LSTM** char LM longer than the smoke config.

| Config | Steps | Size | Goal |
|--------|------:|------|------|
| `llm_smoke` | 200 | ~108k | pipeline check |
| **`llm_toy` (this notebook)** | **5000** | ~4-layer / 128-d | readable-ish text |

**Local:** project `.venv` kernel → Run All.

**Colab (GPU):** Runtime → GPU → Run All.

> **Colab:** clones branch `dev/other` (contains `LLM_spiked`). If the repo folder already exists from an old clone, delete `/content/Spiking-Neural-Network` or run setup's fetch/checkout.

Expect wall time on CPU: roughly **1–3+ hours**. GPU is much faster.

## 1. Environment and repo root

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

IN_COLAB = Path("/content").exists()
REPO_URL = "https://github.com/AlexWoods1/Spiking-Neural-Network.git"
# * Set to a branch/tag that contains LLM_spiked once pushed, e.g. "dev/other".
REPO_REF = "dev/other"

def _has_llm_spiked(root: Path) -> bool:
    return (root / "src" / "spiking_neural_network" / "LLM_spiked" / "model.py").is_file()

if IN_COLAB:
    ROOT = Path("/content/Spiking-Neural-Network")
    if not ROOT.exists():
        !git clone --branch {REPO_REF} {REPO_URL} {ROOT}
    else:
        print("Repo already present:", ROOT)
        %cd {ROOT}
        !git fetch --all
        !git checkout {REPO_REF}
        !git pull --ff-only origin {REPO_REF} || true
    %cd {ROOT}
    # * Colab is usually 3.11/3.12; repo may pin 3.14.
    pyproject = ROOT / "pyproject.toml"
    text = pyproject.read_text(encoding="utf-8")
    if 'requires-python = ">=3.14"' in text:
        pyproject.write_text(
            text.replace('requires-python = ">=3.14"', 'requires-python = ">=3.11"'),
            encoding="utf-8",
        )
        print("Patched requires-python to >=3.11")
    # * Keep Colab's JAX if CUDA already works; otherwise install cuda12 wheels.
    !pip install -q optax pyyaml numpy tqdm
    try:
        import jax
        if not any("cuda" in str(d).lower() or "gpu" in str(d).lower() for d in jax.devices()):
            !pip install -q -U "jax[cuda12]"
    except Exception:
        !pip install -q -U "jax[cuda12]"
else:
    ROOT = Path.cwd()
    if not (ROOT / "src" / "spiking_neural_network").is_dir():
        for candidate in [ROOT, *ROOT.parents]:
            if (candidate / "src" / "spiking_neural_network").is_dir():
                ROOT = candidate
                break
    os.chdir(ROOT)

src = str(ROOT / "src")
# * Prefer this checkout over any older pip-installed package.
sys.path = [p for p in sys.path if "spiking_neural_network" not in p.replace("\\", "/")]
if src not in sys.path:
    sys.path.insert(0, src)

import importlib
import jax

print("ROOT", ROOT)
print("JAX", jax.__version__, "devices", jax.devices())
print("LLM_spiked present:", _has_llm_spiked(ROOT))
if not _has_llm_spiked(ROOT):
    print(
        "\n!!! LLM_spiked missing from this checkout.\n"
        "Run the next cell to upload a local zip bundle, or push LLM_spiked to GitHub\n"
        f"and set REPO_REF to that branch (currently {REPO_REF!r}).\n"
    )
else:
    importlib.invalidate_caches()
    from spiking_neural_network.LLM_spiked.data import CharTokenizer  # noqa: F401
    print("Import OK: spiking_neural_network.LLM_spiked")


Cloning into '/content/Spiking-Neural-Network'...
remote: Enumerating objects: 396, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 396 (delta 24), reused 53 (delta 15), pack-reused 268 (from 1)
Receiving objects: 100% (396/396), 1.01 MiB | 27.94 MiB/s, done.
Resolving deltas: 100% (164/164), done.
/content/Spiking-Neural-Network
Patched requires-python to >=3.11
ROOT /content/Spiking-Neural-Network
JAX 0.7.2 devices [CudaDevice(id=0)]


## 1b. Colab only — upload `LLM_spiked` if GitHub is missing it

On your PC (repo root), create a zip:

```powershell
Compress-Archive -Path src\spiking_neural_network\LLM_spiked,configs,scripts\prepare_shakespeare.py,scripts\train_llm.py -DestinationPath llm_spiked_bundle.zip -Force
```

Then run the next cell and select `llm_spiked_bundle.zip`. Skip this section if setup already printed `Import OK`.

In [ ]:
import io
import shutil
import zipfile
from pathlib import Path

assert "ROOT" in globals(), "Run the setup cell first."

if _has_llm_spiked(ROOT):
    print("LLM_spiked already present — skip upload.")
elif not IN_COLAB:
    raise SystemExit(
        "LLM_spiked missing locally. Build/open this repo on the machine that has the package."
    )
else:
    from google.colab import files

    print("Upload llm_spiked_bundle.zip …")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file uploaded.")
    name, raw = next(iter(uploaded.items()))
    zpath = ROOT / name
    zpath.write_bytes(raw)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        zf.extractall(ROOT / "_bundle_extract")
    extracted = ROOT / "_bundle_extract"

    # * Accept either a nested LLM_spiked/ or src/spiking_neural_network/LLM_spiked/.
    candidates = list(extracted.rglob("LLM_spiked"))
    pkg = next((p for p in candidates if (p / "model.py").is_file()), None)
    if pkg is None:
        raise SystemExit(f"Could not find LLM_spiked/model.py inside {name}")
    dest = ROOT / "src" / "spiking_neural_network" / "LLM_spiked"
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(pkg, dest)

    # Optional extras from the same zip
    for rel in ("configs", "scripts"):
        src_extra = next((p for p in extracted.rglob(rel) if p.is_dir()), None)
        if src_extra is not None:
            for item in src_extra.iterdir():
                target = ROOT / rel / item.name
                target.parent.mkdir(parents=True, exist_ok=True)
                if item.is_file():
                    shutil.copy2(item, target)

    shutil.rmtree(extracted, ignore_errors=True)
    import importlib
    importlib.invalidate_caches()
    from spiking_neural_network.LLM_spiked.data import CharTokenizer  # noqa: F401
    print("Import OK after upload:", dest)


## 2. Longer-run config (`llm_toy`)

Writes `configs/llm_toy.yaml` if missing. Tweak `MAX_STEPS` / `BATCH_SIZE` below for your machine.

In [2]:
# --- knobs (edit these) ---
MAX_STEPS = 5000
BATCH_SIZE = 32          # drop to 16 on low-RAM CPU
N_LAYER, N_HEAD, N_EMBD = 4, 4, 128
BLOCK_SIZE = 256
SAMPLE_INTERVAL = 500    # expensive on CPU; raise to 1000 to go faster
EVAL_INTERVAL = 100
CHECKPOINT_INTERVAL = 1000

CFG_PATH = ROOT / "configs" / "llm_toy.yaml"
CFG_PATH.parent.mkdir(parents=True, exist_ok=True)
CFG_PATH.write_text(
    f"""# Generated by notebooks/train_llm_long.ipynb
model:
  n_layer: {N_LAYER}
  n_head: {N_HEAD}
  n_embd: {N_EMBD}
  block_size: {BLOCK_SIZE}
  vocab_size: 65
  dropout: 0.1
  bias: true
  v_th: 0.5
  leak: 0.5
  v_minus: -1.0
  v_plus: 2.0
  alpha: 1.0
  beta: 1.0

train:
  batch_size: {BATCH_SIZE}
  max_steps: {MAX_STEPS}
  learning_rate: 3.0e-4
  weight_decay: 0.1
  beta1: 0.9
  beta2: 0.99
  warmup_steps: 100
  grad_clip: 1.0
  eval_interval: {EVAL_INTERVAL}
  eval_batches: 10
  sample_interval: {SAMPLE_INTERVAL}
  checkpoint_interval: {CHECKPOINT_INTERVAL}
  seed: 1337

data:
  dataset: shakespeare
  data_dir: data/shakespeare
  train_frac: 0.9

paths:
  out_dir: checkpoints/llm_toy
  tokenizer_path: data/shakespeare/tokenizer.json
""",
    encoding="utf-8",
)
print("Wrote", CFG_PATH)

Wrote /content/Spiking-Neural-Network/configs/llm_toy.yaml


## 3. Prepare Shakespeare + char tokenizer

In [10]:
import importlib.util

from spiking_neural_network.LLM_spiked.data import CharTokenizer

# * Load prepare helpers without requiring scripts/ to be a package.
_prep_path = ROOT / "scripts" / "prepare_shakespeare.py"
_spec = importlib.util.spec_from_file_location("prepare_shakespeare", _prep_path)
_prep = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_prep)

data_dir = ROOT / "data" / "shakespeare"
tok_path = data_dir / "tokenizer.json"

if not (data_dir / "train.txt").is_file() or not tok_path.is_file():
    text = _prep.download_shakespeare()
    train, val = _prep.split_train_val(text, 0.9)
    _prep.write_splits(data_dir, train, val)
    tok = CharTokenizer.from_text(text)
    tok.save(tok_path)
    print(f"Tokenizer vocab_size={tok.vocab_size}")
else:
    tok = CharTokenizer.load(tok_path)
    print(f"Reusing data in {data_dir} (vocab_size={tok.vocab_size})")

ModuleNotFoundError: No module named 'spiking_neural_network.LLM_spiked'

## 4. Train

Checkpoints land in `checkpoints/llm_toy/ckpt_{step}.pkl`.
Healthy progress: val CE falls from ~`ln(65)≈4.17` toward **~2.0 or lower** by a few thousand steps.

In [ ]:
from spiking_neural_network.LLM_spiked.train import train

params = train(CFG_PATH)
print("Training finished. Final param tree keys:", list(params.keys()))

## 5. Generate from the last checkpoint

In [ ]:
from spiking_neural_network.LLM_spiked.generate import generate, load_checkpoint

ckpt_dir = ROOT / "checkpoints" / "llm_toy"
ckpts = sorted(ckpt_dir.glob("ckpt_*.pkl"), key=lambda p: int(p.stem.split("_")[1]))
assert ckpts, f"No checkpoints in {ckpt_dir}"
ckpt = ckpts[-1]
print("Using", ckpt)

params, model_cfg, tok = load_checkpoint(ckpt)
text = generate(
    params,
    tok,
    model_cfg,
    prompt="ROMEO:",
    max_tokens=400,
    temperature=0.7,
    top_k=20,
    seed=0,
)
print(text)

## 6. Colab only — download checkpoint

In [ ]:
if IN_COLAB:
    from google.colab import files

    files.download(str(ckpt))
else:
    print("Local run — checkpoint already at", ckpt)